# MiniLM embedding-space ablations

Test corpus-wide embedding transformations for **`sentence-transformers/all-MiniLM-L6-v2`** while keeping the model, datasets, evaluation definition, and cosine similarity fixed.

The working hypothesis is that some embedding dimensions contribute mostly non-query-specific background or noise. Corpus calibration can suppress common offsets and rescale dimensions according to their typical variability.

This notebook evaluates:

1. standard cosine / identity transform
2. mean-centered cosine
3. variance-normalized cosine
4. z-normalized cosine

Query-adapted weighting and top-k dimension gating are documented at the end as the next scoring experiments because they are query-specific rather than corpus-wide transforms.


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("/content/retrieval-benchlab")
if "google.colab" in sys.modules:
    if not (REPO_ROOT / ".git").exists():
        !git clone --depth 1 https://github.com/lohex/retrieval-benchlab.git {REPO_ROOT}
    else:
        !git -C {REPO_ROOT} pull --ff-only
    %cd {REPO_ROOT}
else:
    REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))

!pip -q install -U datasets sentence-transformers


In [ ]:
import hashlib
import logging

import pandas as pd
import torch
from IPython.display import display
from sentence_transformers import SentenceTransformer

from src.embedding_transforms import EmbeddingTransformConfig, EmbeddingTransformType
from src.evaluate import RuntimeConfig, compute_calibration_statistics, evaluate, register_evaluation, register_pipeline
from src.io import load_calibration_set, mount_google_drive

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", force=True)
logger = logging.getLogger("minilm-ablation")


## Configuration

Calibration uses a fixed document corpus disjoint from the evaluation datasets. Statistics are estimated from raw document embeddings before any L2 normalization.


In [ ]:
DATASETS_ROOT = "/content/drive/MyDrive/Retreaval/data"
CALIBRATION_SET_PATH = "/content/drive/MyDrive/Retreaval/calibration/bioasq-5k"
REGISTRY_DB_PATH = "/content/drive/MyDrive/Retreaval/databases/datasets.sqlite"
RESULTS_DB_PATH = "/content/drive/MyDrive/Retreaval/databases/results.sqlite"

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EPSILON = 1e-6

METRIC_CONFIG = {
    "mrr_at_k": (10,),
    "ndcg_at_k": (10,),
    "accuracy_at_k": (1, 3, 5, 10, 100),
    "precision_recall_at_k": (1, 3, 5, 10, 100),
    "map_at_k": (100,),
}

runtime = RuntimeConfig(batch_size=64, corpus_scan_size=10_000, show_progress_bar=True, device="cuda" if torch.cuda.is_available() else "cpu")
evaluation_id = register_evaluation(METRIC_CONFIG, registry_db_path=REGISTRY_DB_PATH)
logger.info("Evaluation: %s", evaluation_id)
logger.info("Device: %s", runtime.device)


## Compute reproducible calibration statistics

For calibration document embeddings `d_i`, estimate one mean and standard deviation per embedding dimension. The calibration `source_id` is derived from the actual document IDs and text, so provenance is independent of the local path. The complete `mu` and `sigma` vectors are also stored in pipeline identity.


In [ ]:
def calibration_source_id(corpus: dict[str, str]) -> str:
    digest = hashlib.sha256()
    for document_id in sorted(corpus):
        for value in (document_id, corpus[document_id]):
            encoded = value.encode("utf-8")
            digest.update(len(encoded).to_bytes(8, byteorder="big"))
            digest.update(encoded)
    return f"bioasq-5k:{digest.hexdigest()[:24]}"

mount_google_drive()
calibration_set = load_calibration_set(CALIBRATION_SET_PATH)
source_id = calibration_source_id(calibration_set.corpus)
calibration_model = SentenceTransformer(MODEL_NAME, device=runtime.device)
calibration_statistics = compute_calibration_statistics(
    calibration_model,
    calibration_set.corpus.values(),
    source_id=source_id,
    batch_size=runtime.batch_size,
    show_progress_bar=runtime.show_progress_bar,
)
logger.info("Calibration source: %s", source_id)
logger.info("Embedding dimensions: %d", len(calibration_statistics.mean))


## Register MiniLM ablation pipelines

All pipelines use cosine similarity. Only the embedding transformation changes. Cosine normalization occurs after these transformations.


In [ ]:
identity_pipeline_id = register_pipeline(model_name=MODEL_NAME, similarity_metric="cosine", registry_db_path=REGISTRY_DB_PATH)
mean_center_pipeline_id = register_pipeline(model_name=MODEL_NAME, similarity_metric="cosine", embedding_transform=EmbeddingTransformConfig(transform_type=EmbeddingTransformType.MEAN_CENTER, calibration=calibration_statistics, epsilon=EPSILON), registry_db_path=REGISTRY_DB_PATH)
variance_pipeline_id = register_pipeline(model_name=MODEL_NAME, similarity_metric="cosine", embedding_transform=EmbeddingTransformConfig(transform_type=EmbeddingTransformType.VARIANCE_NORMALIZE, calibration=calibration_statistics, epsilon=EPSILON), registry_db_path=REGISTRY_DB_PATH)
z_pipeline_id = register_pipeline(model_name=MODEL_NAME, similarity_metric="cosine", embedding_transform=EmbeddingTransformConfig(transform_type=EmbeddingTransformType.Z_NORMALIZE, calibration=calibration_statistics, epsilon=EPSILON), registry_db_path=REGISTRY_DB_PATH)

pipelines = {
    "MiniLM / cosine": identity_pipeline_id,
    "MiniLM / mean-center + cosine": mean_center_pipeline_id,
    "MiniLM / variance-normalize + cosine": variance_pipeline_id,
    "MiniLM / z-normalize + cosine": z_pipeline_id,
}
pipelines


## Evaluate every latest dataset version

All four variants share the same evaluation definition and datasets. This isolates the effect of embedding-space calibration.


In [ ]:
records = []
for pipeline_name, pipeline_id in pipelines.items():
    outcomes = evaluate(
        pipeline_id=pipeline_id,
        evaluation_id=evaluation_id,
        datasets_root=DATASETS_ROOT,
        runtime=runtime,
        registry_db_path=REGISTRY_DB_PATH,
        results_db_path=RESULTS_DB_PATH,
    )
    for outcome in outcomes:
        record = {
            "pipeline": pipeline_name,
            "dataset": outcome.dataset_name,
            "version": outcome.dataset_version,
            "status": outcome.status.value,
            "dataset_id": outcome.dataset_id,
            "result_id": outcome.result_id,
        }
        if outcome.metrics is not None:
            record.update(outcome.metrics)
        records.append(record)

results_table = pd.DataFrame(records).sort_values(["pipeline", "dataset", "version"])
identifier_columns = {"pipeline", "dataset", "version", "status", "dataset_id", "result_id"}
metric_columns = [column for column in results_table.columns if column not in identifier_columns]
display(results_table.style.format({column: "{:.4f}" for column in metric_columns}, na_rep=""))


## Next scoring experiments: query adaptation and top-k gating

These experiments are intentionally not implemented as corpus-wide `EmbeddingTransformConfig`s because their document weighting depends on the individual query.

### Query-adapted weighted cosine

Start from z-normalized embeddings and derive query-specific weights `w_k = |z_q,k|^alpha`. `alpha = 0` must reproduce the z-normalized reference; values `> 0` progressively strengthen query adaptation.

### Top-k dimension gating

Rank dimensions independently for every query by `|z_q,k|` and retain only the most query-specific dimensions before scoring. Initial gates should include **25%, 50%, and 75%**; a broader sweep such as **5%, 10%, 25%, 50%, 75%, 100%** can test whether quality remains stable or improves while dimensions are removed.

Both methods require a query-dependent scoring layer whose parameters are part of pipeline identity. They should be implemented there rather than as notebook-local scoring code.
